Readme


Cell 1 — Imports

Added transformer but there are already enough variations ask if that's still needed?

In [15]:
import zipfile
import random
import copy
import warnings
import math
from dataclasses import dataclass
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

Cell 2 — Config

In [2]:
@dataclass
class Config:
    zip_path: str = "/Users/lilysong/Desktop/deep learning/Project/CMAPSSData.zip"
    subset: str = "FD001"

    random_seed: int = 42
    train_ratio: float = 0.8

    batch_size: int = 128
    lr: float = 1e-3
    num_epochs: int = 100
    patience: int = 10
    weight_decay: float = 1e-5

    # LSTM baseline
    hidden_size: int = 128
    num_layers: int = 1
    lstm_dropout: float = 0.2

    # DCNN variant
    dcnn_num_filters: int = 32
    dcnn_fc_units: int = 100
    dcnn_dropout: float = 0.2

    window_size: int = 30
    rul_clip: int = 125

    # Transformer variant
    trans_d_model: int = 64
    trans_nhead: int = 4
    trans_num_layers: int = 2
    trans_ff_dim: int = 128
    trans_dropout: float = 0.1

    device: str = "cuda" if torch.cuda.is_available() else "cpu"

CFG = Config()
print("Using device:", CFG.device)

Using device: cpu


Cell 3 — Seed

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CFG.random_seed)

Cell 4 — Load Data

In [4]:
def get_zip_filenames(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        print(z.namelist()[:30])
        return z.namelist()

def resolve_member_path(zip_path, target_name):
    names = get_zip_filenames(zip_path)
    if target_name in names:
        return target_name

    matches = [name for name in names if name.endswith(target_name)]
    if len(matches) == 1:
        return matches[0]
    elif len(matches) > 1:
        raise ValueError(f"Multiple matches found for {target_name}: {matches}")
    else:
        raise FileNotFoundError(f"Could not find {target_name} inside zip.")

def read_cmapss_from_zip(zip_path, filename):
    member_path = resolve_member_path(zip_path, filename)
    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(member_path) as f:
            df = pd.read_csv(f, sep=r"\s+", header=None)
    return df

train_df = read_cmapss_from_zip(CFG.zip_path, f"train_{CFG.subset}.txt")
test_df = read_cmapss_from_zip(CFG.zip_path, f"test_{CFG.subset}.txt")
rul_df = read_cmapss_from_zip(CFG.zip_path, f"RUL_{CFG.subset}.txt")

columns = (
    ["unit", "cycle"] +
    [f"setting_{i}" for i in range(1, 4)] +
    [f"sensor_{i}" for i in range(1, 22)]
)

train_df.columns = columns
test_df.columns = columns
rul_df.columns = ["RUL"]

print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)
print("rul_df shape:", rul_df.shape)

['Damage Propagation Modeling.pdf', 'readme.txt', 'RUL_FD001.txt', 'RUL_FD002.txt', 'RUL_FD003.txt', 'RUL_FD004.txt', 'test_FD001.txt', 'test_FD002.txt', 'test_FD003.txt', 'test_FD004.txt', 'train_FD001.txt', 'train_FD002.txt', 'train_FD003.txt', 'train_FD004.txt']
['Damage Propagation Modeling.pdf', 'readme.txt', 'RUL_FD001.txt', 'RUL_FD002.txt', 'RUL_FD003.txt', 'RUL_FD004.txt', 'test_FD001.txt', 'test_FD002.txt', 'test_FD003.txt', 'test_FD004.txt', 'train_FD001.txt', 'train_FD002.txt', 'train_FD003.txt', 'train_FD004.txt']
['Damage Propagation Modeling.pdf', 'readme.txt', 'RUL_FD001.txt', 'RUL_FD002.txt', 'RUL_FD003.txt', 'RUL_FD004.txt', 'test_FD001.txt', 'test_FD002.txt', 'test_FD003.txt', 'test_FD004.txt', 'train_FD001.txt', 'train_FD002.txt', 'train_FD003.txt', 'train_FD004.txt']
train_df shape: (20631, 26)
test_df shape: (13096, 26)
rul_df shape: (100, 1)


Cell 5 — Preprocessing

In [5]:
RAW_FEATURE_COLS = [f"setting_{i}" for i in range(1, 4)] + [f"sensor_{i}" for i in range(1, 22)]

# compute train RUL
max_cycle = train_df.groupby("unit")["cycle"].max().reset_index()
max_cycle.columns = ["unit", "max_cycle"]
train_df = train_df.merge(max_cycle, on="unit")
train_df["RUL"] = train_df["max_cycle"] - train_df["cycle"]
train_df.drop(columns=["max_cycle"], inplace=True)

# RUL clipping
train_df["RUL"] = train_df["RUL"].clip(upper=CFG.rul_clip)

# engine-level split
all_units = sorted(train_df["unit"].unique())
rng = np.random.RandomState(CFG.random_seed)
rng.shuffle(all_units)

split_idx = int(CFG.train_ratio * len(all_units))
train_units = all_units[:split_idx]
val_units = all_units[split_idx:]

train_split_df = train_df[train_df["unit"].isin(train_units)].copy()
val_split_df = train_df[train_df["unit"].isin(val_units)].copy()

print("Train engines:", len(train_units))
print("Val engines:", len(val_units))

# remove near-zero variance features
feature_var = train_split_df[RAW_FEATURE_COLS].var()
feature_cols = feature_var[feature_var > 1e-8].index.tolist()
dropped_cols = [c for c in RAW_FEATURE_COLS if c not in feature_cols]

print("Selected features:", len(feature_cols))
print("Dropped features:", dropped_cols)

# standardization
scaler = StandardScaler()
train_split_df[feature_cols] = scaler.fit_transform(train_split_df[feature_cols])
val_split_df[feature_cols] = scaler.transform(val_split_df[feature_cols])
test_df[feature_cols] = scaler.transform(test_df[feature_cols])

Train engines: 80
Val engines: 20
Selected features: 17
Dropped features: ['setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']


Cell 6 — Build Sequences

In [6]:
def build_train_sequences(df, feature_cols, target_col, window_size):
    X_list, y_list = [], []

    for unit_id in sorted(df["unit"].unique()):
        unit_df = df[df["unit"] == unit_id].sort_values("cycle")
        features = unit_df[feature_cols].values
        targets = unit_df[target_col].values

        if len(unit_df) < window_size:
            continue

        for i in range(len(unit_df) - window_size + 1):
            X_window = features[i:i + window_size]
            y_target = targets[i + window_size - 1]
            X_list.append(X_window)
            y_list.append(y_target)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    return X, y

def build_test_sequences(test_df, rul_df, feature_cols, window_size):
    X_list, y_list = [], []

    for i, unit_id in enumerate(sorted(test_df["unit"].unique())):
        unit_df = test_df[test_df["unit"] == unit_id].sort_values("cycle")
        features = unit_df[feature_cols].values

        if len(features) < window_size:
            pad_len = window_size - len(features)
            pad = np.repeat(features[0:1], pad_len, axis=0)
            features = np.vstack([pad, features])

        X_window = features[-window_size:]
        y_target = min(float(rul_df.iloc[i]["RUL"]), CFG.rul_clip)

        X_list.append(X_window)
        y_list.append(y_target)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    return X, y

X_train, y_train = build_train_sequences(train_split_df, feature_cols, "RUL", CFG.window_size)
X_val, y_val = build_train_sequences(val_split_df, feature_cols, "RUL", CFG.window_size)
X_test, y_test = build_test_sequences(test_df, rul_df, feature_cols, CFG.window_size)

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

Train: (14020, 30, 17) (14020,)
Val:   (3711, 30, 17) (3711,)
Test:  (100, 30, 17) (100,)


Cell 7 — Dataset / Dataloader

In [7]:
class CMAPSSDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(CMAPSSDataset(X_train, y_train), batch_size=CFG.batch_size, shuffle=True)
val_loader = DataLoader(CMAPSSDataset(X_val, y_val), batch_size=CFG.batch_size, shuffle=False)
test_loader = DataLoader(CMAPSSDataset(X_test, y_test), batch_size=CFG.batch_size, shuffle=False)

Cell 8 — Metrics

In [8]:
def rmse_score(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def phm_score(y_true, y_pred):
    d = y_pred - y_true
    score = 0.0
    for di in d:
        if di < 0:
            score += np.exp(-di / 13.0) - 1.0
        else:
            score += np.exp(di / 10.0) - 1.0
    return float(score)

Cell 9 — Models

In [9]:
class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.2):
        super().__init__()
        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=lstm_dropout,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)              # [B, T, H]
        last_out = out[:, -1, :]           # [B, H]
        last_out = self.dropout(last_out)
        y_hat = self.fc(last_out)          # [B, 1]
        return y_hat.squeeze(1)


class DCNNRegressor(nn.Module):
    """
    DCNN for C-MAPSS RUL prediction
    More paper-like version:
    - 5 temporal conv layers
    - no pooling
    - flatten + fully connected layers
    """
    def __init__(self, input_size, window_size, num_filters=32, fc_units=100, dropout=0.5):
        super().__init__()

        self.window_size = window_size
        self.num_filters = num_filters

        # Input after transpose: [B, F, T]
        self.conv1 = nn.Conv1d(input_size, num_filters, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(num_filters, num_filters, kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(num_filters, num_filters, kernel_size=3, padding=1)
        self.conv4 = nn.Conv1d(num_filters, num_filters, kernel_size=3, padding=1)
        self.conv5 = nn.Conv1d(num_filters, num_filters, kernel_size=3, padding=1)

        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        # flatten after conv stack: [B, num_filters, T] -> [B, num_filters * T]
        self.fc1 = nn.Linear(num_filters * window_size, fc_units)
        self.fc2 = nn.Linear(fc_units, 1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        # x: [B, T, F] -> [B, F, T]
        x = x.transpose(1, 2)

        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = self.act(self.conv3(x))
        x = self.act(self.conv4(x))
        x = self.act(self.conv5(x))

        # flatten instead of global average pooling
        x = x.reshape(x.size(0), -1)

        x = self.dropout(x)
        x = self.act(self.fc1(x))
        y_hat = self.fc2(x)

        return y_hat.squeeze(1)

Cell 10 — Train / Evaluate Functions

In [10]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        preds = model(X_batch)
        loss = criterion(preds, y_batch)

        total_loss += loss.item() * X_batch.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(y_batch.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    return {
        "loss": avg_loss,
        "rmse": rmse_score(all_targets, all_preds),
        "mae": float(mean_absolute_error(all_targets, all_preds)),
        "phm": phm_score(all_targets, all_preds),
        "preds": all_preds,
        "targets": all_targets
    }


def fit_model(model, train_loader, val_loader, cfg):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_rmse": []
    }

    best_val_rmse = float("inf")
    best_epoch = -1
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, cfg.num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, cfg.device)
        val_metrics = evaluate(model, val_loader, criterion, cfg.device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_rmse"].append(val_metrics["rmse"])

        print(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val RMSE: {val_metrics['rmse']:.4f}"
        )

        if val_metrics["rmse"] < best_val_rmse:
            best_val_rmse = val_metrics["rmse"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= cfg.patience:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)
    return model, history, best_epoch

Cell 11 — Run One Model

In [11]:
def run_model(model_name, cfg):
    set_seed(cfg.random_seed)

    if model_name == "LSTM":
        model = LSTMRegressor(
            input_size=len(feature_cols),
            hidden_size=cfg.hidden_size,
            num_layers=cfg.num_layers,
            dropout=cfg.lstm_dropout
        ).to(cfg.device)

    elif model_name == "DCNN":
        model = DCNNRegressor(
            input_size=len(feature_cols),
            window_size=cfg.window_size,
            num_filters=cfg.dcnn_num_filters,
            fc_units=cfg.dcnn_fc_units,
            dropout=cfg.dcnn_dropout
        ).to(cfg.device)

    else:
        raise ValueError("Unknown model name")

    print(model)

    model, history, best_epoch = fit_model(model, train_loader, val_loader, cfg)

    criterion = nn.MSELoss()
    val_metrics = evaluate(model, val_loader, criterion, cfg.device)
    test_metrics = evaluate(model, test_loader, criterion, cfg.device)

    summary = {
        "model": model_name,
        "best_epoch": best_epoch,
        "val_rmse": val_metrics["rmse"],
        "val_mae": val_metrics["mae"],
        "val_phm": val_metrics["phm"],
        "test_rmse": test_metrics["rmse"],
        "test_mae": test_metrics["mae"],
        "test_phm": test_metrics["phm"],
    }
    return model, history, summary, test_metrics


Cell 12 — Run Main Comparison

In [ ]:
lstm_model, lstm_history, lstm_summary, lstm_test = run_model("LSTM", CFG)
dcnn_model, dcnn_history, dcnn_summary, dcnn_test = run_model("DCNN", CFG)
os.makedirs("checkpoints", exist_ok=True)

torch.save(lstm_model.state_dict(), "checkpoints/lstm_model.pt")
torch.save(dcnn_model.state_dict(), "checkpoints/dcnn_model.pt")

print("All checkpoints saved.")

LSTMRegressor(
  (lstm): LSTM(17, 128, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)
Epoch 001 | Train Loss: 6669.5604 | Val Loss: 5815.4958 | Val RMSE: 76.2594
Epoch 002 | Train Loss: 4695.3504 | Val Loss: 4188.4297 | Val RMSE: 64.7181
Epoch 003 | Train Loss: 3358.1216 | Val Loss: 2959.7538 | Val RMSE: 54.4036
Epoch 004 | Train Loss: 2246.2403 | Val Loss: 1882.0964 | Val RMSE: 43.3831
Epoch 005 | Train Loss: 1419.2685 | Val Loss: 1239.9890 | Val RMSE: 35.2135
Epoch 006 | Train Loss: 968.4386 | Val Loss: 883.6251 | Val RMSE: 29.7258
Epoch 007 | Train Loss: 696.0794 | Val Loss: 661.8439 | Val RMSE: 25.7263
Epoch 008 | Train Loss: 542.6752 | Val Loss: 534.3298 | Val RMSE: 23.1156
Epoch 009 | Train Loss: 430.6045 | Val Loss: 448.4073 | Val RMSE: 21.1756
Epoch 010 | Train Loss: 356.0789 | Val Loss: 411.6177 | Val RMSE: 20.2884
Epoch 011 | Train Loss: 309.8132 | Val Loss: 349.4490 | Val RMSE: 18.6936
Epoch 012 | Tra

NameError: name 'trans_summary' is not defined

Cell 13 — Graph 1: RMSE Comparison

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(main_df["model"], main_df["test_rmse"])
plt.xlabel("Model")
plt.ylabel("Test RMSE")
plt.title("RMSE Comparison on FD001")
plt.grid(axis="y")
plt.show()

Cell 14 — Graph 2: PHM Comparison

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(main_df["model"], main_df["test_phm"])
plt.xlabel("Model")
plt.ylabel("Test PHM Score")
plt.title("PHM Score Comparison on FD001")
plt.grid(axis="y")
plt.show()

Cell 15 — Graph 3: Training Curves for Both Models

In [ ]:
# 1) Train / Val Loss comparison
plt.figure(figsize=(8, 5))
plt.plot(lstm_history["train_loss"], label="LSTM Train Loss")
plt.plot(lstm_history["val_loss"], label="LSTM Val Loss")
plt.plot(dcnn_history["train_loss"], label="DCNN Train Loss")
plt.plot(dcnn_history["val_loss"], label="DCNN Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss: LSTM vs DCNN")
plt.legend()
plt.grid(True)
plt.show()

# 2) Validation RMSE comparison
plt.figure(figsize=(8, 5))
plt.plot(lstm_history["val_rmse"], label="LSTM Val RMSE")
plt.plot(dcnn_history["val_rmse"], label="DCNN Val RMSE")
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.title("Validation RMSE: LSTM vs DCNN")
plt.legend()
plt.grid(True)
plt.show()

Cell 16 — Final Print

In [ ]:
print(main_df)